In [ ]:
El ejemplo se puede descargar de : https://github.com/apastorini/sprites_label/tree/main


### 📝 Objetivos de la Clase

  * Comprender la importancia de una **arquitectura de juego modular** (separando las clases en módulos y paquetes).
  * Aprender a implementar **controles de animación basados en la entrada del usuario** (teclas de flecha).
  * Modificar las clases `GameObject` y `Player` para que la animación no sea automática, sino que avance o retroceda un fotograma a la vez.

-----

### 🏛️ Estructura del Proyecto

Para mantener el código organizado, usaremos una estructura de directorios modular.

```
tu_proyecto/
├── assets/
│   ├── sprites/
│   │   ├── player_animation/
│   │   │   ├── player.json
│   │   │   └── spritesheet.png
│   │   └── ... (otros sprites)
├── engine/
│   ├── __init__.py
│   ├── game_object.py
│   └── game.py
├── main.py
```

  * **`engine/`**: Un paquete que contendrá las clases centrales del "motor" de tu juego, como los objetos base y el bucle principal.
  * **`main.py`**: El archivo que se ejecutará para iniciar el juego.
  * **`assets/`**: Directorio para todos los recursos del juego (imágenes, sonidos, etc.).

-----

### 👨‍🏫 Parte 1: El Cerebro de la Animación (`game_object.py`)

La clase `GameObject` es la base para cualquier entidad que pueda animarse en tu juego. A diferencia de tu código original, modificaremos el método `update` para que **no avance automáticamente** los fotogramas. En su lugar, el `Player` se encargará de decidir cuándo cambiar el fotograma basándose en la entrada del usuario.

**Archivo: `engine/game_object.py`**

```python
# engine/game_object.py
import pygame
import os
import json
import collections

# Colores y configuraciones del juego (se mantienen aquí por simplicidad)
ROJO = (255, 0, 0)
AZUL_OSCURO = (0, 0, 50)
BLANCO = (255, 255, 255)
NEGRO = (0, 0, 0)

class GameObject:
    def __init__(self, x, y, json_path):
        self.x = x
        self.y = y
        self.sprite_sheet = None
        self.animations = {}
        self.current_animation = 'idle'
        self.frame_index = 0
        self.rect = None
        self.load_from_json(json_path)

    def load_from_json(self, json_path):
        """
        Carga la spritesheet y los datos de animación desde un archivo JSON.
        Ahora maneja la ruta relativa de forma segura.
        """
        try:
            # os.path.dirname obtiene la carpeta del JSON
            base_dir = os.path.dirname(os.path.abspath(json_path))
            with open(json_path, 'r') as f:
                data = json.load(f)

            # Usamos os.path.join para construir una ruta multiplataforma segura
            spritesheet_path_relative = data.get('spritesheet_path')
            # Quitamos la ruta absoluta que tenías en el JSON para hacerlo portable
            spritesheet_path = os.path.join(base_dir, os.path.basename(spritesheet_path_relative))
            
            # Cargar la imagen
            self.sprite_sheet = pygame.image.load(spritesheet_path).convert_alpha()
            
            self.animations = collections.defaultdict(list)
            for anim_name, anim_list in data.get('animations', {}).items():
                for sprite_data in anim_list:
                    # Guardamos las coordenadas de cada sprite como una tupla
                    self.animations[anim_name].append(
                        (sprite_data['x'], sprite_data['y'], sprite_data['width'], sprite_data['height'])
                    )
            
            if not self.animations:
                raise ValueError("No se encontraron animaciones válidas en el JSON.")

            # Establecer la animación inicial y el rectángulo de colisión
            self.current_animation = 'giro' if 'giro' in self.animations else list(self.animations.keys())[0]
            first_frame_rect_data = self.animations[self.current_animation][0]
            self.rect = pygame.Rect(self.x, self.y, first_frame_rect_data[2], first_frame_rect_data[3])

        except FileNotFoundError:
            print(f"Error: El archivo JSON o la spritesheet no fue encontrado en '{json_path}'.")
            self.sprite_sheet = None
            self.animations = {}
        except Exception as e:
            print(f"Error al cargar el JSON: {e}")
            self.sprite_sheet = None
            self.animations = {}

    def draw(self, surface):
        """
        Dibuja el fotograma actual en la pantalla.
        """
        if not self.sprite_sheet or not self.animations or not self.rect: return
            
        frame_rect_data = self.animations[self.current_animation][self.frame_index]
        # Usamos subsurface para extraer el sprite correcto de la hoja
        frame = self.sprite_sheet.subsurface(pygame.Rect(frame_rect_data))
        surface.blit(frame, self.rect)

```

-----

### 👨‍🏫 Parte 2: La Lógica del Personaje (`game.py`)

Aquí es donde entra la clase `Player`,que  ahora controlará directamente el índice del fotograma en la animación.

**Archivo: `engine/player.py`**

```python
# engine/player.py
import pygame
import os
import random
from engine.game_object import GameObject

# Inicializar Pygame
pygame.init()

class Player(GameObject):
    def __init__(self, x, y):
        # La ruta al JSON de nuestro personaje
        json_path = os.path.join('assets', 'sprites', 'player_animation', 'player.json')
        super().__init__(x, y, json_path=json_path)
        self.velocidad = 5

    def mover(self, keys):
        """
        Controla el movimiento y la animación del jugador.
        """
        if keys[pygame.K_LEFT]:
            # Retroceder en los fotogramas
            self.frame_index -= 1
            # Si llegamos al inicio, volver al final
            if self.frame_index < 0:
                self.frame_index = len(self.animations[self.current_animation]) - 1

        if keys[pygame.K_RIGHT]:
            # Avanzar en los fotogramas
            self.frame_index += 1
            # Si llegamos al final, volver al inicio
            if self.frame_index >= len(self.animations[self.current_animation]):
                self.frame_index = 0

        # No hay movimiento horizontal, solo cambio de fotograma.
        self.rect.x = self.x
        self.rect.y = self.y

```

```python
#Archivo: `engine/game.py`**
import pygame

from engine.game_object import AZUL_OSCURO
from engine.player import Player


class Game:
    def __init__(self):
        self.ANCHO = 800
        self.ALTO = 600
        self.PANTALLA = pygame.display.set_mode((self.ANCHO, self.ALTO))
        pygame.display.set_caption('Spritesheet con Teclas')

        self.jugador = Player(350, 250)
        self.reloj = pygame.time.Clock()
        self.estado_juego = 'jugando'

    def game_loop(self):
        ejecutando = True
        while ejecutando:
            # Obtener el tiempo transcurrido, aunque no lo usemos para la animación
            # es una buena práctica para mantener el bucle constante.
            dt = self.reloj.tick(15) # Menor FPS para ver mejor el cambio de frame
            
            # --- INPUT: Siempre procesar eventos ---
            for evento in pygame.event.get():
                if evento.type == pygame.QUIT:
                    ejecutando = False
            
            # --- Lógica del juego ---
            if self.estado_juego == 'jugando':
                teclas = pygame.key.get_pressed()
                self.jugador.mover(teclas)
                # No llamamos a update, ya que el movimiento se encarga de la animación
            
            # --- Renderizado ---
            self.PANTALLA.fill(AZUL_OSCURO)
            
            if self.estado_juego == 'jugando':
                self.jugador.draw(self.PANTALLA)
            
            pygame.display.flip()

```

-----

### 👨‍🏫 Parte 3: El Inicio del Juego (`main.py`)

Este archivo es el punto de entrada a tu juego. Su única función es crear una instancia de la clase `Game` y comenzar el bucle principal.

**Archivo: `main.py`**

```python
# main.py
import pygame
from engine.game import Game

if __name__ == '__main__':
    # Creación y ejecución del juego
    game = Game()
    game.game_loop()

    # Salir de Pygame al terminar
    pygame.quit()

```

### 👨‍🏫 Parte 4: El Archivo JSON (`player.json`)

Para que el código funcione, tu archivo JSON debe ser portable. Es decir, no debe contener rutas absolutas como `C:/TVJ/clase 5 -editor/assets/spritesheet.png`. En su lugar, el `spritesheet_path` debe ser una ruta relativa a la ubicación del archivo JSON.

**Archivo: `assets/sprites/player_animation/player.json`**

```json
{
    "project_name": "giro",
    "spritesheet_path": "spritesheet.png",
    "character_name": "mio",
    "animations": {
        "giro": [
            {
                "x": 2, "y": 5, "width": 59, "height": 55,
                "name": "sprite_0", "description": "Sugerido", "animation": "giro"
            },
            {
                "x": 74, "y": 6, "width": 46, "height": 54,
                "name": "sprite_1", "description": "Sugerido", "animation": "giro"
            },
            {
                "x": 150, "y": 6, "width": 19, "height": 54,
                "name": "sprite_2", "description": "Sugerido", "animation": "giro"
            },
            {
                "x": 201, "y": 6, "width": 45, "height": 54,
                "name": "sprite_3", "description": "Sugerido", "animation": "giro"
            },
            {
                "x": 259, "y": 5, "width": 59, "height": 55,
                "name": "sprite_4", "description": "Sugerido", "animation": "giro"
            },
            {
                "x": 331, "y": 6, "width": 43, "height": 54,
                "name": "sprite_5", "description": "Sugerido", "animation": "giro"
            },
            {
                "x": 406, "y": 6, "width": 20, "height": 54,
                "name": "sprite_6", "description": "Sugerido", "animation": "giro"
            },
            {
                "x": 458, "y": 6, "width": 43, "height": 54,
                "name": "sprite_7", "description": "Sugerido", "animation": "giro"
            }
        ]
    }
}
```

### 🧠 Beneficios de este Enfoque

  * **Mayor Control:** Al no depender de un `frame_timer` automático, tienes control total sobre la velocidad y el flujo de la animación. Esto es ideal para acciones específicas como un ataque, un salto o, en este caso, girar un fotograma a la vez.
  * **Desacoplamiento:** La lógica de la animación (`GameObject`) está separada de la lógica del jugador (`Player`). El `GameObject` solo sabe cómo mostrar un fotograma, mientras que el `Player` sabe cuándo y qué fotograma mostrar.
  * **Reusabilidad:** La clase `GameObject` se puede reutilizar para cualquier entidad del juego (enemigos, coleccionables, etc.) que use spritesheets, sin importar cómo se controle su animación.
  * **Organización:** La estructura de paquetes y directorios mantiene el código limpio y fácil de mantener, especialmente a medida que el proyecto crece.